# 複数のプロンプトをまとめてバッチ生成する

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/mamorlis/nlpcookbook/blob/main/notebooks/12_llm/llm_batch_generation.ipynb)

NLP クックブック 12-05 ／ 大規模言語モデル (LLM)

大量の入力を LLM で処理したい。1 件ずつ `generate` すると遅いので、バッチでまとめて生成したい。

In [1]:
import time
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

name = "sbintuitions/sarashina2.2-0.5b-instruct-v0.1"
tokenizer = AutoTokenizer.from_pretrained(name)
model = AutoModelForCausalLM.from_pretrained(name).eval()
tokenizer.padding_side = "left"               # 生成モデルのバッチ処理では必ず左パディング
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

words = ["形態素解析", "機械翻訳", "固有表現抽出", "要約", "質問応答", "感情分析", "音声認識", "情報検索"]
prompts = [tokenizer.apply_chat_template([{"role": "user", "content": f"「{w}」を一文で説明してください。"}],
                                         add_generation_prompt=True, tokenize=False) for w in words]

@torch.no_grad()
def generate(batch):
    enc = tokenizer(batch, return_tensors="pt", padding=True, add_special_tokens=False)
    out = model.generate(**enc, max_new_tokens=40, do_sample=False, pad_token_id=tokenizer.pad_token_id)
    return tokenizer.batch_decode(out[:, enc["input_ids"].shape[1]:], skip_special_tokens=True)

t = time.perf_counter(); single = [generate([p])[0] for p in prompts]; t1 = time.perf_counter() - t
t = time.perf_counter(); batched = generate(prompts); t2 = time.perf_counter() - t
print(f"1 件ずつ: {t1:.1f} 秒 / バッチ: {t2:.1f} 秒")
for w, s in zip(words, batched):
    print(f"{w}: {s.strip()}")

1 件ずつ: 269.0 秒 / バッチ: 106.7 秒
形態素解析: 「形態素解析」とは、文章中の単語やフレーズをその構成要素である「形態素」に分割し、それぞれの意味や品詞を解析する処理のことを指します。これにより、文章の意味を
機械翻訳: 機械翻訳とは、コンピュータープログラムを用いて言語を翻訳する技術です。人間の翻訳者が手動で行う作業を自動化することで、大量のテキストを迅速に翻訳することが可能になります。これにより、ビジネスコミュニケーションや学術研究など
固有表現抽出: 「固有表現抽出」とは、テキストデータから特定の意味を持つ単語やフレーズ（固有表現）を抽出するプロセスを指します。これにより、テキスト内の重要な情報を抽出し、分析や検索を容易にすることができます
要約: 「要約」とは、ある文章や文書を簡潔にまとめ、要点を抽出するプロセスを指します。これにより、重要な情報を効率的に理解し、議論や分析のベースとすることができます。
質問応答: 「質問応答」とは、ユーザーに対して特定の質問に対して正確かつ有用な回答を提供するプロセスを指します。このプロセスでは、データベースやAIモデルを用いて質問を分析し、適切な回答を生成します。これにより、ユーザーは必要な情報
感情分析: 感情分析は、テキストデータから感情を特定し、その感情の強度やニュアンスを分析する技術です。これにより、文章がポジティブかネガティブか、あるいは特定の感情（例えば喜び、悲しみ、怒り
音声認識: 音声認識とは、音声をデジタル信号に変換し、コンピュータがそれを認識してテキストやコマンドとして解釈する技術です。これにより、音声入力や音声コマンドによる操作が可能になり、様々なアプリケーションやデバイス
情報検索: 情報検索とは、特定のテーマや質問に対して、大量の情報の中から効率的に探し出し、解釈するプロセスを指します。これはコンピュータや人手を用いて行われることが多く、必要な情報を迅速に得るための重要な手段です。


- 右パディングのままだと、短いプロンプトの後ろにパディングが入り、その続きを生成することになって出力が壊れる。
- バッチ内の長さを揃える (長さでソートする) とパディングが減って効率が良い。
- 数万件以上を本格的に処理するなら、vLLM などの推論サーバー (連続バッチ処理) が桁違いに速い。